12/20/2022 Spaceship Titanic Competition Project

This code is a bite more complicated than the other one. Pipeline and OneHotEncoding were used. Therefore the codes are longer than just using get_dummies and impute.

Many thanks to Kaggler ROLAND ERIKSSON for sharing his notebook. 


# ***Loading and Previewing Data***

In [1]:
import pandas as pd


# Load the train data and test data
spaceship_file_path = '../input/spaceship-titanic/train.csv'
spaceship_data = pd.read_csv(spaceship_file_path)

test_file_path = '../input/spaceship-titanic/test.csv'
test_data = pd.read_csv(test_file_path)

display(spaceship_data.head())
display(test_data.head())

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


In [2]:
print("The number of variables in the original data:")
spaceship_data.columns
spaceship_data.dtypes

The number of variables in the original data:


PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

In [3]:
#print("The number of variables in the test data:")
#test_data.columns
print("The variable data type:")
test_data.dtypes
#print("The shape of the data ")
#test_data.shape

The variable data type:


PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
dtype: object

In [4]:
test_data.isnull().sum()

PassengerId       0
HomePlanet       87
CryoSleep        93
Cabin           100
Destination      92
Age              91
VIP              93
RoomService      82
FoodCourt       106
ShoppingMall     98
Spa             101
VRDeck           80
Name             94
dtype: int64

In [5]:
#print("The number of variables in the train data:")
#spaceship_data.columns
print("The variable data type in the train:")
spaceship_data.dtypes
#print("The shape of the data ")
#spaceship_data.shape

The variable data type in the train:


PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

In [6]:
spaceship_data.Age.describe()

count    8514.000000
mean       28.827930
std        14.489021
min         0.000000
25%        19.000000
50%        27.000000
75%        38.000000
max        79.000000
Name: Age, dtype: float64

In [7]:
spaceship_data.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [8]:
spaceship_data.duplicated().sum()

0

In [9]:
test_data.duplicated().sum()

0

In [10]:
# check the cardinality of the train data and the test data
spaceship_data.nunique()

PassengerId     8693
HomePlanet         3
CryoSleep          2
Cabin           6560
Destination        3
Age               80
VIP                2
RoomService     1273
FoodCourt       1507
ShoppingMall    1115
Spa             1327
VRDeck          1306
Name            8473
Transported        2
dtype: int64

In [11]:
test_data.nunique()

PassengerId     4277
HomePlanet         3
CryoSleep          2
Cabin           3265
Destination        3
Age               79
VIP                2
RoomService      842
FoodCourt        902
ShoppingMall     715
Spa              833
VRDeck           796
Name            4176
dtype: int64

In [12]:
# These codes are from a Kaggler:ROLAND ERIKSSON
# The reason is that in my original codes 'Cabin' and 'PassengerId' were dropped, 
# Therefore we may lose some important information. In this new version, we
# used Roland's code to extract some information from 'Cabin' and 'PassenerId'.

# Split Cabin column
def split_cabin(df):
    newcols = df["Cabin"].str.split("/", expand=True)
    newcols.index = df.index
    df["Deck"] = newcols.iloc[:,0]
    df["Side"] = newcols.iloc[:,2]

# Add passenger group id and group size
def add_groupid(df):
    splitdf = df["PassengerId"].str.split("_", expand=True)
    df["GroupId"] = splitdf.iloc[:, 0]
    
def add_groupsize(df):
    grpsizes = df.groupby("GroupId").size()
    newcol = grpsizes[df["GroupId"]]
    newcol.index = df.index
    df["GroupSize"] = newcol.astype(float)   
    
# Combine all column manipulation
def preprocess(df):
    split_cabin(df)
    add_groupid(df)
    add_groupsize(df)
    
preprocess(spaceship_data)
preprocess(test_data)

In [13]:
spaceship_data.head()
spaceship_data.shape

(8693, 18)

In [14]:
test_data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Deck,Side,GroupId,GroupSize
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning,G,S,0013,1.0
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers,F,S,0018,1.0
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus,C,S,0019,1.0
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter,C,S,0021,1.0
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez,F,S,0023,1.0


# Preparing for Machine Learning

In [15]:
features = ['HomePlanet', 'CryoSleep','Destination','Age','VIP',
            "RoomService","FoodCourt","ShoppingMall","Spa","VRDeck",
           'Deck', 'Side', 'GroupSize']

# Select columns corresponding to features, and preview the data
X_train_full = spaceship_data[features]
X_test_full = test_data[features]

In [16]:
X_test_full.shape

(4277, 13)

In [17]:
X_train_full.shape

(8693, 13)

In [18]:
# Imports for preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Select categorical columns with relatively low cardinality 
categorical_cols = [cname for cname in X_train_full.columns if 
                   X_train_full[cname].nunique()<10 and 
                                X_train_full[cname].dtype == "object"]
#categorical_cols
# Select numerical columns
numerical_cols = [cname for cname in X_train_full.columns if 
                   X_train_full[cname].dtype in ['int64','float64']]
#numerical_cols

# Keep selected columns only
my_cols = categorical_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_test = X_test_full[my_cols].copy()
#X_train.shape
#X_test.shape

#Set up for preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy='median')

#Set up for preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('onehot',OneHotEncoder(handle_unknown='ignore'))
])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(transformers=[('num',numerical_transformer,numerical_cols),
                                              ('cat',categorical_transformer,categorical_cols)])




In [19]:
categorical_cols
numerical_cols

['Age',
 'RoomService',
 'FoodCourt',
 'ShoppingMall',
 'Spa',
 'VRDeck',
 'GroupSize']

# Machine Learning

In [20]:
# define model

from xgboost import XGBClassifier

my_model = XGBClassifier(n_estimators = 100, learning_rate = 0.05,max_depth=5,random_state=0)

# Bundle preprocessing and modeling code in a pipeline
spacet=Pipeline(steps=[('preprocessor',preprocessor),
                      ('model',my_model)])
#set the target
y = spaceship_data.Transported

# fit the model
spacet.fit(X_train,y)

# preprocessing of the test data, get predictions
# This model outputs boolean 0 or 1
my_list = spacet.predict(X_test)

In [21]:
my_list

array([1, 0, 1, ..., 1, 1, 1])

In [22]:
# We want the output to be False or True. 
predictions = [bool(item) for item in my_list]

#Use cross validation to evaluate the performance of the model
from sklearn.model_selection import cross_val_score
scores = cross_val_score(spacet, X_train, y, cv=5)
print("Mean cross-validation score: %.4f" % scores.mean())


Mean cross-validation score: 0.7977


In [23]:
# output predictions data
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Transported': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
